# Run a simulation

**Model:** `pfc_models.models.logpfc`  
**Written against:** pfc-models 0.0.0  
**Takeaway:** how to build a GPU geometry, set model parameters, step the solver and view the density field

Migrated 2026-10-12 from `log/gpu-spectral-base-implementation/_testSimulation.ipynb`. Code cells are carried across unchanged apart from import rewrites; stored outputs were stripped, so re-run to regenerate the figures.

> **Seeding.** The hot initial-condition generators now draw from `np.random.default_rng(seed)` instead of seeding global NumPy state. Call sites are unchanged, but a given integer seed produces a different noise field than it did in the source fork.


In [ ]:
# numerics
import numpy as np
import cupy as cp

# plotting
import plotly.io as pio

# Set the global default renderer
pio.renderers.default = "png"

from importlib import reload

In [ ]:
# density plotting
import pfc_models.viz.heatmap as _densityHeatmap
reload(_densityHeatmap)
from pfc_models.viz.heatmap import *

In [ ]:
# Initial conditions
import pfc_models.fields.initial_conditions_2d as _sim_initial_conditions
reload(_sim_initial_conditions)
from pfc_models.fields.initial_conditions_2d import *

In [ ]:
# Simulation
import pfc_models.models.logpfc.simulation as Simulation
reload(Simulation)
from pfc_models.models.logpfc.simulation import *

# Simulation Builder

In [ ]:
# geometry for verifiable derivatives
Nx = 66*4
Ny = 66*4
q0_star = 1.0
a = 4*np.pi/3**0.5 / q0_star
h = 4*np.pi / q0_star
Mx = 7*4
My = 4*4
Lx = Mx * a
Ly = My * h

# model parameters
Gamma_val = 1.0
beta_val = 0.5
q0 = q0_star
epsilon_val = -0.9
g_val = -2.0
meanDensity_S = 0.55
meanDensity_V = 0.3
relStd = 0.1
kMax = np.sqrt(3) / 2
V0 = -1e-3
potWidth = 10.0
dt = 1e-1

geometry = Build2DGeometry(Nx=Nx, Ny=Ny, Lx=Lx, Ly=Ly)
modelParameters = BuildLogPFCModelParameters(
  Gamma=Gamma_val,
  beta=beta_val,
  q0=q0,
  epsilon=epsilon_val,
  g=g_val,
  dt=dt,
  meanDensity=meanDensity_S
)

sim = BuildCupyGPU_Simulation(modelParameters, geometry)
# phiInit = GeneratePerfectCrystalInitialWithCentralVacancy(geometry, meanDensity_S, meanDensity_V, A0=0.15, q0=q0)
# phiInit = GeneratePerfectCrystalInitialConditions(geometry, meanDensity_S, A0=0.19)
phiInit = GenerateHotInitialConditions(geometry, meanDensity_S, relStd, kMax)
sim.State.phi[...] = cp.asarray(phiInit, dtype=cpFloatType)
print(
    geometry.X.shape,
    geometry.KX.shape,
    geometry.X[0,1], geometry.Y[1,0],
  )

X, Y = sim.Geometry.X, sim.Geometry.Y
GetDensityHeatmap(sim.phi.get(), X, Y, title='ϕ', showScale=True).show(renderer="notebook")

# Test Simulation

In [ ]:
totalT = 1000.0
steps = 100
miniSteps = int(totalT / steps / sim.Model.dt)
print(f'Running {steps} steps of {miniSteps} mini-steps each (totalT = {totalT})')
for i in range(steps):
  for _ in range(miniSteps):
    sim.TimeStep()
  f = sim.CalculateFreeEnergy()
  print(f'Step {i}: f = {f:0.15f}, ϕ.min = {sim.phi.get().min()}, ϕ.max = {sim.phi.get().max()}, ϕ.mean = {sim.phi.get().mean()}')

print(f'Step {i+1}: f = {f:0.15f}, ϕ.min = {sim.phi.get().min()}, ϕ.max = {sim.phi.get().max()}, ϕ.mean = {sim.phi.get().mean()}')

GetDensityHeatmap(sim.State.phi_next.get(), X, Y, title='ϕ(t + ΔT)', showScale=True).show()

In [ ]:
GetDensityHeatmap(sim.Get_local_phi(), X, Y, title='⟨ϕ⟩', ppx=4, showScale=True).show(renderer="notebook")